# LLM & RAG Workshop (SUT)

© 2026 Parinya Duangklang — https://github.com/parinyad123/LLM-and-RAG-Workshop-SUT-

This notebook (code) is licensed under the **MIT License**. See `LICENSE-CODE`.  
Workshop slides are licensed under **CC BY-NC-SA 4.0** (see `LICENSE-CONTENT`).  
You may reuse and adapt with credit; not for commercial use without permission.

#Part 0: SETUP

## 0.1 Install Libraries

In [1]:
!pip install -q \
    langchain-community \
    langchain-core \
    langchain-groq \
    chromadb \
    faiss-cpu \
    sentence-transformers \
    gradio \
    pypdf \
    langchain-huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/14

## 0.2 Import Libraries

In [2]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader, PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_community.vectorstores import Chroma, FAISS

from langchain_groq import ChatGroq

import os
import time
import numpy as np
from pathlib import Path
import re

from google.colab import userdata, files

import gradio as gr

## 0.3 Configure API keys and initialize LLM

In [3]:
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    llm = ChatGroq(
        model = "openai/gpt-oss-120b",
        # model="llama-3.3-70b-versatile"
        temperature=0.2
    )
    print("Using Groq")
except:
    print("Groq key not found!")
    print("Please add GROQ_API_KEY to Colab Secrets\n")

Using Groq


## 0.4 Initialize Embeddings



In [4]:
# --- Embedding Model Configuration ---
embeddings = HuggingFaceEmbeddings(
    # model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_name="sentence-transformers/all-mpnet-base-v2",
    # model_kwargs={'device': 'cuda'},
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print(f"Model loaded")

# --- System Testing ---
print("\nTesting embeddings...")

# Convert a sample text string into a Vector (a long sequence of numerical data)
test_vec = embeddings.embed_query(
    "SpaceX, founded in 2002, is worth $800 billion based on a private tender offer in December 2025"
)

# Verify system functionality by checking the Vector length
print(f"Embeddings working! (dim={len(test_vec)})")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded

Testing embeddings...
Embeddings working! (dim=768)


# Part 1: Document Preparation

## 1.1 Data Ingestion

In [5]:
# Download instructional materials
!gdown 1m1OI12Hrcs4kOJ8i56_iq4xGQ84QFoFz -O Newton-biography.pdf

# Use PyPDFLoader (a LangChain utility) to read and parse the PDF file
pdf_loader = PyPDFLoader("Newton-biography.pdf")

# Load content into the application
newton_docs = pdf_loader.load()

# Print the total number of pages loaded from the PDF to confirm success
print(f"Load successful! Total pages: {len(newton_docs)}")

# Show a snippet of the first page to verify content
print(f"Preview of the first page: {newton_docs[0].page_content[:100]}...")

Downloading...
From: https://drive.google.com/uc?id=1m1OI12Hrcs4kOJ8i56_iq4xGQ84QFoFz
To: /content/Newton-biography.pdf
100% 26.9k/26.9k [00:00<00:00, 12.7MB/s]
Load successful! Total pages: 6
Preview of the first page: Sir Isaac Newton 
 
Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England  
Died: 31 March 1727 in ...


## 1.2 Validate PDF quality

In [6]:
def inspect_document(docs: list) -> None:

    # Merge all pages into a single string for global analysis
    full_text = " ".join([d.page_content for d in docs])
    total_chars = len(full_text)

    print("=== Document Health Check ===")
    print(f"Pages           : {len(docs)}")
    print(f"Total chars     : {total_chars}")

    # Handle edge case: empty document
    if total_chars == 0:
        print("\n⚠️ Empty document — nothing to inspect.")
        return

    # Count formatting issues: Null bytes, triple spaces, and triple newlines
    null_bytes = full_text.count('\x00')
    excess_spaces = len(re.findall(r' {3,}', full_text))
    excess_lines  = len(re.findall(r'\n{3,}', full_text))

    # Display global summary with status indicators
    print(f"\nNull bytes      : {null_bytes}   {'⚠️ Action: Clean needed' if null_bytes > 0 else '✅ None found'}")
    print(f"Excess spaces   : {excess_spaces}   {'⚠️ Action: Clean needed' if excess_spaces > 0 else '✅ None found'}")
    print(f"Excess newlines : {excess_lines}   {'⚠️ Action: Clean needed' if excess_lines > 0 else '✅ None found'}")

    # Analyze each page individually to pinpoint issues
    print("\n--- Per-page Breakdown ---")
    for i, doc in enumerate(docs):
        p = doc.page_content
        flags = []
        # Check specific flags per page
        if p.count('\x00')              > 0: flags.append("null bytes")
        if len(re.findall(r' {3,}', p)) > 0: flags.append("excess spaces")
        if len(re.findall(r'\n{3,}', p))> 0: flags.append("excess newlines")

        status = ("⚠️  [" + ", ".join(flags) + "]") if flags else "✅ Clean"
        print(f"  Page {i+1}: {status}")

    # Aggregate all detected issues for the final verdict
    issues = []
    if null_bytes    > 0: issues.append("Null bytes")
    if excess_spaces > 0: issues.append("Excess spaces")
    if excess_lines  > 0: issues.append("Excess newlines")

    print()
    if issues:
        print(f"Conclusion: ⚠️ Cleaning recommended — Issues: {', '.join(issues)}")
    else:
        print("Conclusion: ✅ Document looks clean")

In [7]:
inspect_document(newton_docs)

=== Document Health Check ===
Pages           : 6
Total chars     : 22468

Null bytes      : 0   ✅ None found
Excess spaces   : 0   ✅ None found
Excess newlines : 0   ✅ None found

--- Per-page Breakdown ---
  Page 1: ✅ Clean
  Page 2: ✅ Clean
  Page 3: ✅ Clean
  Page 4: ✅ Clean
  Page 5: ✅ Clean
  Page 6: ✅ Clean

Conclusion: ✅ Document looks clean


In [8]:
def clean_document(docs: list) -> list:

    cleaned_docs = []

    for doc in docs:
        text = doc.page_content

        # 1. Remove Null bytes (binary artifacts often found in PDFs/scans)
        text = text.replace('\x00', '')

        # 2. Collapse large clusters of spaces (3+) into a single space
        text = re.sub(r' {3,}', ' ', text)

        # 3. Normalize vertical whitespace: Reduce excessive newlines (3+) to double newlines
        text = re.sub(r'\n{3,}', '\n\n', text)

        # 4. Convert single line-wraps (\n) into spaces
        text = text.replace('\n\n', '<<PARA>>')
        text = text.replace('\n', ' ')
        text = text.replace('<<PARA>>', '\n\n')

        # 5. Clean up any double spaces that may have been created during conversion
        text = re.sub(r' {2,}', ' ', text)

        # Reconstruct Document with sanitized content and original metadata
        cleaned_docs.append(
            Document(
                page_content=text,
                metadata=doc.metadata
            )
        )

    return cleaned_docs

In [9]:
# clean Newton document
cleaed_newton_docs = clean_document(newton_docs)

In [10]:
cleaed_newton_docs

[Document(metadata={'producer': 'Amyuni PDF Converter', 'creator': 'PyPDF', 'creationdate': '1/2/2005 2:32:43', 'title': 'Microsoft Word - Newton-bio.doc', 'version': 'Version 2.09 Pro - Developer Licence # 14BAD6A1-1B9E', 'source': 'Newton-biography.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton\'s life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge. The third period (nearly as long as the other two combined) saw Newton as a highly paid government official in London with little further interest in mathematical research. Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the calendar in use at th

## 1.3 Text Splitting

In [11]:
# Document Splitting with RecursiveCharacterTextSplitter
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=250,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Apply the splitter to the cleaned documents (cleaed_newton_docs)
newton_chunks = recursive_splitter.split_documents(cleaed_newton_docs)

print(f"{len(newton_chunks)} Chunks")

83 Chunks


In [12]:
newton_chunks

[Document(metadata={'producer': 'Amyuni PDF Converter', 'creator': 'PyPDF', 'creationdate': '1/2/2005 2:32:43', 'title': 'Microsoft Word - Newton-bio.doc', 'version': 'Version 2.09 Pro - Developer Licence # 14BAD6A1-1B9E', 'source': 'Newton-biography.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content="Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge"),
 Document(metadata={'producer': 'Amyuni PDF Converter', 'creator': 'PyPDF', 'creationdate': '1/2/2005 2:32:43', 'title': 'Microsoft Word - Newton-bio.doc', 'version': 'Version 2.09 Pro - Developer Licence # 14BAD6A1-1B9E', 'source': 'Newton-biography.pdf', 'total_pages': 6, 'page': 0, 'page_label

# Part 2: Vector Database

Remove Vector database

In [13]:
import os
import shutil

# Define the list of directories used for storing vector databases
dirs_to_remove = ["./chroma_db", "./faiss_db", "./faiss_index"]

# Iterate through the list to check and delete each directory
for dir_path in dirs_to_remove:
    if os.path.exists(dir_path):
        shutil.rmtree(dir_path)
        print(f"✅ Deleted: {dir_path}")
    else:
        print(f"⚠️  Not found: {dir_path}")

print("\nDone — ready to rebuild vector stores!")

⚠️  Not found: ./chroma_db
⚠️  Not found: ./faiss_db
⚠️  Not found: ./faiss_index

Done — ready to rebuild vector stores!


## 2.1 FAISS

### 2.1.1 สร้าง Vectorstore

In [14]:

# Start timer (to demonstrate the performance efficiency of FAISS to learners)
start_time = time.time()

# Create the Vector Store using FAISS
vectorstore_faiss = FAISS.from_documents(
    documents=newton_chunks,
    embedding=embeddings
)

# Calculate the time taken to build the index
faiss_build_time = time.time() - start_time
print(f"FAISS build time: {faiss_build_time:.3f}s")

# .index.ntotal retrieves the total count of vectors (or chunks) stored in the database
print(f"Total vectors: {vectorstore_faiss.index.ntotal}")

FAISS build time: 28.589s
Total vectors: 83


### 2.1.2 Basic Search + Score

In [15]:
# Define the query to search for within the documents
query = "When and where was Newton born?"

# Basic Similarity Search
faiss_results = vectorstore_faiss.similarity_search(query, k=5)

# Search with Relevance Scoring
faiss_results_scored = vectorstore_faiss.similarity_search_with_score(query, k=5)

print("=== FAISS (L2 Distance: Lower = Better) ===")
for doc, score in faiss_results_scored:
    print(f"Score: {score:.4f} | {doc.page_content[:150]}...")

=== FAISS (L2 Distance: Lower = Better) ===
Score: 0.4813 | . Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the calendar in use at the time of his birth he ...
Score: 0.5309 | Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into...
Score: 0.5652 | . The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive peri...
Score: 0.6255 | . Although by the calendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 in this biography whi...
Score: 0.7708 | . (The Gregorian calendar was not adopted in England until 1752.) Isaac Newton came from a family of farmers but never knew his father, also named Isa...


In [16]:
faiss_results[0]

Document(id='9feaa297-bf4d-4e96-af55-25b6c28cb5c6', metadata={'producer': 'Amyuni PDF Converter', 'creator': 'PyPDF', 'creationdate': '1/2/2005 2:32:43', 'title': 'Microsoft Word - Newton-bio.doc', 'version': 'Version 2.09 Pro - Developer Licence # 14BAD6A1-1B9E', 'source': 'Newton-biography.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='. Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the calendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 in this biography which is the "corrected" Gregorian calendar date bringing it into line with our present calendar')

In [17]:
faiss_results_scored[0]

(Document(id='9feaa297-bf4d-4e96-af55-25b6c28cb5c6', metadata={'producer': 'Amyuni PDF Converter', 'creator': 'PyPDF', 'creationdate': '1/2/2005 2:32:43', 'title': 'Microsoft Word - Newton-bio.doc', 'version': 'Version 2.09 Pro - Developer Licence # 14BAD6A1-1B9E', 'source': 'Newton-biography.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='. Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the calendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 in this biography which is the "corrected" Gregorian calendar date bringing it into line with our present calendar'),
 np.float32(0.48130345))

### 2.1.3 Index Inspection & Types

In [18]:
# Access the underlying FAISS index directly
raw_index = vectorstore_faiss.index

# Verify the Index Type
print(f"Index type   : {type(raw_index)}")

# Check the total number of Vectors (Chunks) stored in memory
print(f"Total vectors: {raw_index.ntotal}")

# Verify the Vector Dimensions (Dimensionality)
print(f"Dimension    : {raw_index.d}")

Index type   : <class 'faiss.swigfaiss_avx2.IndexFlatL2'>
Total vectors: 83
Dimension    : 768


In [19]:
raw_index.reconstruct(1)

array([-1.83132174e-03,  6.10201154e-03, -2.18350403e-02,  4.32466343e-02,
        1.94438007e-02, -4.25631553e-02, -8.02172255e-03, -5.47219021e-03,
       -7.85819534e-03, -3.65617201e-02,  3.31173055e-02,  3.48308235e-02,
        2.31983662e-02, -1.12613127e-01,  3.50639261e-02,  2.41436716e-02,
       -2.14665104e-02,  1.02679133e-02, -6.17721491e-02,  1.52501622e-02,
       -3.81899811e-02, -5.44415936e-02,  6.52043149e-03,  1.66473854e-02,
       -5.33888291e-04, -2.98997629e-02, -1.91075616e-02, -2.69712918e-02,
        9.77026392e-03,  5.61679304e-02,  2.17685197e-03,  5.13844006e-02,
        8.48216936e-03,  2.37054620e-02,  1.81120595e-06,  4.82173823e-02,
        3.51513252e-02, -2.24124733e-02,  2.38735043e-02, -1.98442861e-03,
       -2.72741448e-03,  8.57611820e-02,  1.32558839e-02,  2.99110394e-02,
       -3.54086831e-02,  2.46572159e-02, -2.92282756e-02, -7.62999132e-02,
       -1.05287276e-01,  1.90926827e-02, -9.35332384e-03, -6.22358136e-02,
        3.73361930e-02,  

Transitioning to HNSW Index for Large-Scale Retrieval

In [20]:
import faiss

# Get the dimensionality from the current index
dimension  = raw_index.d
# Initialize the HNSW (Hierarchical Navigable Small World) index
hnsw_index = faiss.IndexHNSWFlat(dimension, 32)  # M=32
print(f"Index type   : {type(hnsw_index)}")

# Reconstruct vectors
vectors = np.array([raw_index.reconstruct(i) for i in range(raw_index.ntotal)])
hnsw_index.add(vectors)

print(f"HNSW ready: {hnsw_index.ntotal} vectors")

Index type   : <class 'faiss.swigfaiss_avx2.IndexHNSWFlat'>
HNSW ready: 83 vectors


### 2.1.4 Persistence


In [21]:
# Save the FAISS index to a local directory named "faiss_db"
vectorstore_faiss.save_local("./faiss_db")
print("Database saved to local storage.")

# Loading the Database
vectorstore_faiss_loaded = FAISS.load_local(
    "./faiss_db",
    embeddings,
    allow_dangerous_deserialization=True
)
print("Database loaded successfully.")

Database saved to local storage.
Database loaded successfully.


In [22]:
# Verify Data Retrieval After Reload
reloaded = vectorstore_faiss_loaded.similarity_search(query, k=5)

print(f"Reload successful: {len(reloaded)} results\n")


for i, doc in enumerate(reloaded, 1):
    print(f"{i: >3}. {doc.page_content}")

Reload successful: 5 results

  1. . Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the calendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 in this biography which is the "corrected" Gregorian calendar date bringing it into line with our present calendar
  2. Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge
  3. . The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge. The third period (nearly as long as the o

### 2.1.5 Benchmark

In [23]:
def benchmark_search(vectorstore, query, k=5, runs=100):
    times = []
    for _ in range(runs):
        start = time.time()

        # Execute the actual similarity search query
        vectorstore.similarity_search(query, k=k)
        times.append(time.time() - start)
    return {
        "avg_ms": np.mean(times) * 1000,
        "min_ms": np.min(times) * 1000,
        "max_ms": np.max(times) * 1000
    }

# Execute benchmark for the FAISS vector store
faiss_bench = benchmark_search(vectorstore_faiss, query)

# Display performance results
print(f"Avg: {faiss_bench['avg_ms']:.2f}ms | Min: {faiss_bench['min_ms']:.2f}ms | Max: {faiss_bench['max_ms']:.2f}ms")

Avg: 90.44ms | Min: 76.91ms | Max: 137.26ms


A/B Testing (Flat Index / HNSW Index)

In [24]:
# --- 1. Prepare Both Index Types ---

# Flat Index (Original index from vectorstore_faiss)
flat_index = vectorstore_faiss.index
print(f"Index type   : {type(flat_index)}")

# HNSW Index (The newly created optimized index)
vectorstore_hnsw = FAISS(
    embedding_function=embeddings,
    index=hnsw_index,
    docstore=vectorstore_faiss.docstore,
    index_to_docstore_id=vectorstore_faiss.index_to_docstore_id
)

print(f"Index type   : {type(hnsw_index)}")

# --- 2. Conduct Performance Benchmark ---
runs = 100
print(f"Starting performance benchmark ({runs} iterations)...")
flat_bench = benchmark_search(vectorstore_faiss, query, runs=runs)
hnsw_bench = benchmark_search(vectorstore_hnsw, query, runs=runs)

# --- 3. Display Comparison Results in a Readable Table ---
print("\n" + "="*50)
print(f"{'Index Type':<15} | {'Avg Latency (ms)':<20} | {'Status'}")
print("-"*50)
print(f"{'Flat (L2)':<15} | {flat_bench['avg_ms']:>15.2f} ms | {'Original'}")
print(f"{'HNSW':<15} | {hnsw_bench['avg_ms']:>15.2f} ms | {'Optimized'}")
print("="*50)

# --- 4. Calculate Search Speedup ---
speedup = flat_bench['avg_ms'] / hnsw_bench['avg_ms']
print(f"HNSW is approximately {speedup:.2f}x faster than Flat index!")

Index type   : <class 'faiss.swigfaiss_avx2.IndexFlatL2'>
Index type   : <class 'faiss.swigfaiss_avx2.IndexHNSWFlat'>
Starting performance benchmark (100 iterations)...

Index Type      | Avg Latency (ms)     | Status
--------------------------------------------------
Flat (L2)       |           83.51 ms | Original
HNSW            |           90.54 ms | Optimized
HNSW is approximately 0.92x faster than Flat index!


## 2.2 Chroma

### 2.2.1 Create Vector store

In [25]:
vectorstore_chroma = Chroma.from_documents(
    documents=newton_chunks,
    embedding=embeddings,
    collection_name="newton_biography",
    persist_directory="./chroma_db"
)

# Verify the total number of Vectors currently stored in this Collection
print(f"Total vectors: {vectorstore_chroma._collection.count()}")

Total vectors: 83


### 2.2.2 Basic Search + Score

In [26]:
query = "When and where was Newton born?"

# Basic Similarity Search
# Returns the top 5 (k=5) most relevant Document chunks based on content
chroma_results = vectorstore_chroma.similarity_search(query, k=5)

# Search with Relevance Scoring
# Returns both the content and a "Score" indicating how closely the data matches the query
chroma_results_scored = vectorstore_chroma.similarity_search_with_score(query, k=5)

for doc, score in chroma_results_scored:
    print(f"Score: {score:.4f} | {doc.page_content}")

Score: 0.4813 | . Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the calendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 in this biography which is the "corrected" Gregorian calendar date bringing it into line with our present calendar
Score: 0.5309 | Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge
Score: 0.5652 | . The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge. The third period (nearly as long as th

In [27]:
faiss_results_scored = vectorstore_faiss.similarity_search_with_score(query, k=5)

print("=== FAISS (L2 Distance: Lower = Better) ===")
for doc, score in faiss_results_scored:
    print(f"Score: {score:.4f} | {doc.page_content}")

=== FAISS (L2 Distance: Lower = Better) ===
Score: 0.4813 | . Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the calendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 in this biography which is the "corrected" Gregorian calendar date bringing it into line with our present calendar
Score: 0.5309 | Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge
Score: 0.5652 | . The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambr

# Part 3: Retrieval Strategies

## 1. Basic Retriever — "Baseline"

In [28]:
# Initializing the Retriever
basic_retriever = vectorstore_chroma.as_retriever(
    search_kwargs={"k": 5}
)

# Testing Data Retrieval
query = "What did Newton discover?"
# .invoke() is the standard method to trigger the Retriever to find matches based on the query.
results = basic_retriever.invoke(query)

print(f"Query  : {query}")

for i, doc in enumerate(results, 1):
    print(f"  [{i}]. {doc.page_content}")

Query  : What did Newton discover?
  [1]. . The mechanics of the Copernican astronomy of Galileo attracted him and he also studied Kepler's Optics. He recorded his thoughts in a book which he entitled Quaestiones Quaedam Philosophicae (Certain Philosophical Questions). It is a fascinating account of how Newton's ideas were already forming around 1664. He headed the text with a Latin statement meaning "Plato is my friend, Aristotle is my friend, but my best friend is truth" showing himself a free thinker from an early stage
  [2]. . In July 1669 Barrow tried to ensure that Newton's mathematical achievements became known to the world. He sent Newton's text De Analysi to Collins in London writing:- [Newton] brought me the other day some papers, wherein he set down methods of calculating the dimensions of magnitudes like that of Mr Mercator concerning the hyperbola, but very general; as also of resolving equations; which I suppose will please you; and I shall send you them by the next
  [3

In [29]:
basic_retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x782748081d00>, search_kwargs={'k': 5})

Defining Hard Fact Queries for Accuracy Testing

In [30]:
# Defining Hard Fact Queries for Accuracy Testing
basic_queries = [
    "When and where was Newton born?",           # Expected: 4 Jan 1643, Woolsthorpe
    "What was Newton's role at the Royal Mint?", # Expected: Warden 1696, Master 1699
    "Who was Newton's mother and stepfather?",   # Expected: Hannah Ayscough, Barnabas Smith
]

# Convert the Vector Store into a 'Retriever' interface.
basic_retriever = vectorstore_chroma.as_retriever(search_kwargs={"k": 3})

for query in basic_queries:
    # Trigger the retriever to find context related to the specific fact query.
    results = basic_retriever.invoke(query)

    print(f"{query}")

    for i, doc in enumerate(results, 1):
        # Extract the page number from Metadata
        page = doc.metadata.get('page', '?')
        preview = doc.page_content.replace('\n', ' ')
        print(f"   [{i}] Page {page}: {preview}")

When and where was Newton born?
   [1] Page 0: . Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the calendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 in this biography which is the "corrected" Gregorian calendar date bringing it into line with our present calendar
   [2] Page 0: Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge
   [3] Page 0: . The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge. The third

## 2. Similarity Score Threshold — "Quality Gate"

In [31]:
test_query = "What did Newton discover?"

# Experimenting with strictness levels from 0.3 (Lenient) to 0.9 (Very Strict)
for threshold in [0.3, 0.5, 0.7, 0.9]:
    retriever = vectorstore_chroma.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 15, "score_threshold": threshold}
    )

    # Attempt to retrieve data based on the current threshold
    # Note: LangChain may trigger a warning if no documents meet the criteria.
    results = retriever.invoke(test_query)

    # Create a simple visual bar to show how many chunks passed the filter
    bar = "█" * len(results) # Full Block: Alt+219 |
    print(f"Threshold: {threshold} | Passed: {len(results):>1} chunks | {bar}")


Threshold: 0.3 | Passed: 15 chunks | ███████████████
Threshold: 0.5 | Passed: 12 chunks | ████████████


Threshold: 0.7 | Passed: 0 chunks | 


Threshold: 0.9 | Passed: 0 chunks | 


In [32]:
# Practical Session: Testing 'Out-of-Document' Queries
print("Testing with 'Out-of-Document' Queries")
print("-" * 60)

# Applying a 0.5 threshold (A standard 'Safe' industry benchmark)
safe_retriever = vectorstore_chroma.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.5}
)

test_queries = [
    "When was Newton born?",                # In document (Expected: Pass)
    "What was Newton's favorite pizza?",    # Not in document (Expected: Block)
]

for q in test_queries:
    # The retriever will only return chunks that meet the 0.7 similarity requirement
    res = safe_retriever.invoke(q)
    status = "✅ Found" if res else "⛔ Blocked (Safe from Hallucination)"
    print(f"Query: {q}\nStatus: {status} ({len(res)} chunks)\n")

Testing with 'Out-of-Document' Queries
------------------------------------------------------------
Query: When was Newton born?
Status: ✅ Found (4 chunks)



Query: What was Newton's favorite pizza?
Status: ⛔ Blocked (Safe from Hallucination) (0 chunks)



## 3. Maximal Marginal Relevance (MMR) — "Diversity"

In [33]:
# Comparison: Basic vs. MMR

# query = "Tell me about Newton's life and work"
query = "When and where was Newton born?"

# Basic: Pure Similarity Search (Focuses only on finding the closest matches)
basic_retriever = vectorstore_chroma.as_retriever(search_kwargs={"k": 5})

# MMR: Diversity-Optimized Search (Balances relevance with variety)
mmr_retriever = vectorstore_chroma.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": .5 # 0.5 balance (Sweet Spot)
    }
)

# Execute Retrieval
chunks_basic = basic_retriever.invoke(query)
chunks_mmr = mmr_retriever.invoke(query)

# Compare Results

print("------- BASIC (Similarity) --------")
# Observations: Likely to show highly redundant or "clustered" content.
for i, doc in enumerate(chunks_basic, 1):
    print(f"{i:<3} {doc.page_content}")

print("\n------- MMR (Diversity) --------")
# Observations: Attempts to surface different facets of the topic in the top 5.
for i, doc in enumerate(chunks_mmr, 1):
    print(f"{i:<3} {doc.page_content}")

------- BASIC (Similarity) --------
1   . Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the calendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 in this biography which is the "corrected" Gregorian calendar date bringing it into line with our present calendar
2   Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge
3   . The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge. The third period (nearly as long as th

In [34]:
# Analyzing Content Shifts Based on Lambda

# query = "Tell me about Newton's life and work"
query = "When and where was Newton born?"

# Testing 3 key values:
# 1.0 (Pure Relevance), 0.5 (Balanced), and 0.0 (Maximum Diversity)

for lam in [1.0, 0.5, 0.0]:
    print(f"\nLambda = {lam} " + ("(Pure Relevance)" if lam==1.0 else "(Balanced)" if lam==0.5 else "(Pure Diversity)"))
    print("-" * 30)

    # Note: We fetch 5 chunks to see how the 'neighborhood' of data expands or shrinks
    r = vectorstore_chroma.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": lam}
    )
    results = r.invoke(query)

    for i, doc in enumerate(results, 1):
        print(f"   Chunk {i}: {doc.page_content}")


Lambda = 1.0 (Pure Relevance)
------------------------------
   Chunk 1: . Isaac Newton was born in the manor house of Woolsthorpe, near Grantham in Lincolnshire. Although by the calendar in use at the time of his birth he was born on Christmas Day 1642, we give the date of 4 January 1643 in this biography which is the "corrected" Gregorian calendar date bringing it into line with our present calendar
   Chunk 2: Sir Isaac Newton Born: 4 Jan 1643 in Woolsthorpe, Lincolnshire, England Died: 31 March 1727 in London, England Isaac Newton's life can be divided into three quite distinct periods. The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at Cambridge
   Chunk 3: . The first is his boyhood days from 1643 up to his appointment to a chair in 1669. The second period from 1669 to 1687 was the highly productive period in which he was Lucasian professor at

#Part 4: Complete RAG Pipeline

Query → Retrieve → Format → LLM → Answer

## 4.2 Retriever

In [35]:
# We use Maximum Marginal Relevance (MMR) to balance relevance with information diversity.
# This prevents the LLM from receiving redundant information.
rag_retriever = vectorstore_chroma.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.5
    }
)

print("Retriever : MMR (k=5, fetch_k=20, lambda=0.5)")

Retriever : MMR (k=5, fetch_k=20, lambda=0.5)


## 4.3 Document Formatter

In [36]:
def format_docs(docs):
    """
    Combines multiple document chunks into a single string for the LLM.
    Includes source tracking metadata to improve accuracy and provide citations.
    """
    formatted = []
    for i, doc in enumerate(docs, 1):
        # Extract metadata for transparency; default to '?' if not present
        page   = doc.metadata.get('page', '?')
        source = doc.metadata.get('source', 'Newton Biography')

        # Structure each chunk with a clear header for the LLM to understand
        formatted.append(
            f"[Source {source} | Page {page}]\n{doc.page_content}"
        )

    # Merge all formatted chunks with double newlines for clear separation
    return "\n\n".join(formatted)

print("Formatter : format_docs with source tracking enabled")

Formatter : format_docs with source tracking enabled


## 4.4 Prompt Template

In [37]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant answering questions about Isaac Newton.

Instructions:
- Answer based MAINLY on the provided context
- Be concise and accurate
- If the answer is not in the context, say "I don't have enough information"
- Reference sources naturally (e.g. "According to Source 1...")

Context:
{context}"""),
    ("user", "{question}")
])

print("Prompt : System + User template")

Prompt : System + User template


## 4.5 Build RAG Chain

In [38]:

# We use LCEL (LangChain Expression Language) to pipe components together.

rag_chain = (
    {
        # Fetch relevant chunks & format them
        "context" : rag_retriever | format_docs,

        # Pass the user's question as is
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("Chain built : Retriever → Prompt → LLM → Parser")

Chain built : Retriever → Prompt → LLM → Parser


## 4.6 Test RAG Chain

In [39]:

# Define a list of queries to test both existing facts and out-of-context topics.
test_queries = [
    "When and where was Newton born?",
    "What did Newton discover during the plague years?",
    "What was Newton's relationship with Hooke?",
    "What was Newton's diet and daily routine?",   # Hallucination check
]

for query in test_queries:
    print(f"\nQuery: {query}")

    # Execute the chain. This triggers: Retrieval -> Formatting -> Prompting -> LLM.
    answer = rag_chain.invoke(query)
    print(f"Answer: {answer}")


Query: When and where was Newton born?
Answer: Isaac Newton was born at the manor house in Woolsthorpe, near Grantham in Lincolnshire.  By the calendar then in use his birth fell on Christmas Day 1642, but the biography records the corrected Gregorian date as 4 January 1643【Source Newton‑biography.pdf | Page 0】.

Query: What did Newton discover during the plague years?
Answer: During the two years of the plague (1665‑1667), the young Newton made several groundbreaking discoveries. Most notably, he concluded that **white light is not a simple, indivisible entity but is composed of a spectrum of colors**, a insight he reached by studying chromatic aberration in telescope lenses【Source Newton‑biography.pdf | Page 2】. In the same period he also began “revolutionary advances in mathematics, optics, physics, and astronomy” before he was even 25【Source Newton‑biography.pdf | Page 2】.

Query: What was Newton's relationship with Hooke?
Answer: Newton’s relationship with Robert Hooke was hostil

## 4.7 Debug

In [40]:

def rag_with_debug(query):

    print(f"Query: {query}")
    print("="*60)

    # Manually invoke the retriever to see the raw materials
    docs = rag_retriever.invoke(query)
    print(f"Retrieved {len(docs)} chunks:")
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get('page', '?')
        preview = doc.page_content
        print(f"   [{i}] Page {page}: {preview}")

    # Generate answer
    print(f"\nAnswer:")
    # Execute the final chain to see how the LLM synthesizes the above chunks
    answer = rag_chain.invoke(query)
    print(answer)
    # return answer

# Testing debug mode
rag_with_debug("What were Newton's three laws of motion?")
print("--"*60)
rag_with_debug("What was Newton's relationship with Hooke?")

Query: What were Newton's three laws of motion?
Retrieved 5 chunks:
   [1] Page 3: . Newton's greatest achievement was his work in physics and celestial mechanics, which culminated in the theory of universal gravitation. By 1666 Newton had early versions of his three laws of motion. He had also discovered the law giving the centrifugal force on a body moving uniformly in a circular path. However he did not have a correct understanding of the mechanics of circular motion
   [2] Page 4: . Further generalisation led Newton to the law of universal gravitation:- ... all matter attracts all other matter with a force proportional to the product of their masses and inversely proportional to the square of the distance between them. Newton explained a wide range of previously unrelated phenomena: the eccentric orbits of comets, the tides and their variations, the precession of the Earth's axis, and motion of the Moon as perturbed by the gravity of the Sun
   [3] Page 5: . Given the rage that New

## 4.8 Streaming Response


In [41]:
# Enhances UX by displaying the answer chunk-by-chunk in real-time.

query = "Describe Newton's greatest scientific achievements"
print(f"Query :{query}")
print("Response : ", end="", flush=True)

# Use .stream() instead of .invoke() to get an iterator of text chunks.
for chunk in rag_chain.stream(query):
    print(chunk, end="", flush=True)

Query :Describe Newton's greatest scientific achievements
Response : Newton’s most celebrated scientific accomplishment was the development of his theory of universal gravitation, which unified the physics of falling bodies with the motions of the heavens.  This work was built on his earlier breakthroughs in mechanics—by 1666 he already had early versions of the three laws of motion and a law describing the centrifugal force on a body moving uniformly in a circle, even though his understanding of circular motion was still imperfect【Source Newton‑biography.pdf | Page 3】.  

In addition to his work on gravitation and dynamics, Newton made major contributions to optics.  His delayed but influential treatise *Opticks* (1704) presented a theory of light and colour, explained phenomena such as the colours of thin sheets, “Newton’s rings,” and diffraction, and even combined a wave‑theory element with his corpuscular model【Source Newton‑biography.pdf | Page 3】.  

Thus, Newton’s greatest scien

## 4.9 Batch Processing

In [42]:
# Processes multiple queries simultaneously.

batch_queries = [
    "When was Newton born?",
    "What is the Principia?",
    "When did Newton die?",
]

# Use .batch() to send all queries at once.
answers = rag_chain.batch(batch_queries)

# Match queries with their respective answers using zip().
for query, answer in zip(batch_queries, answers):
    print(f"Query: {query}")
    print(f"Response: {answer}")
    print("--"*60)

Query: When was Newton born?
Response: Isaac Newton was born on 4 January 1643 (according to the corrected Gregorian calendar). He was originally recorded as being born on Christmas Day 1642 under the calendar then used in England.【Source Newton‑biography.pdf | Page 0】
------------------------------------------------------------------------------------------------------------------------
Query: What is the Principia?
Response: The *Principia*—formally titled **Philosophiae Naturalis Principia Mathematica**—is Isaac Newton’s landmark 1687 work that presents a full treatment of his new physics and its application to astronomy.  It is widely regarded as the greatest scientific book ever written and contains Newton’s analysis of the motion of bodies under centripetal forces in both resisting and non‑resisting media【Source Newton-biography.pdf | Page 4】.
------------------------------------------------------------------------------------------------------------------------
Query: When did N

# Part 5: Gradio

## 5.1 Basic Chat Interface

In [43]:

def ask_newton(question):

    # Basic validation: Alert the user if the input is empty or just spaces.
    if not question.strip():
        return "Please enter a question"

    # Pass the question into our RAG Chain
    answer = rag_chain.invoke(question)

    return answer

# Creating the Web Interface using Gradio's High-level 'Interface' class.
demo_basic = gr.Interface(
    fn=ask_newton,             # Function to call on submit
    inputs=gr.Textbox(         # Input component
        label="Your Question",
        placeholder="Ask anything about Isaac Newton...",
        lines=5
    ),
    outputs=gr.Textbox(        # Output component
        label="Answer",
        lines=5
    ),
    title="🍎 Newton Biography RAG",
    description="Ask questions about Isaac Newton using RAG",
    examples=[                # Pre-defined example buttons
        "When and where was Newton born?",
        "What did Newton discover during the plague years?",
        "What was Newton's relationship with Hooke?",
    ]
)

# Launch the local server.
demo_basic.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://349d626fa25d83f787.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 5.2 Advanced Interface with Source Tracking

In [44]:

def ask_newton_with_sources(question):

    if not question.strip():
        return "Please enter a question", ""

    # Manually invoke the retriever to see which chunks were found.
    docs = rag_retriever.invoke(question)

    # Run the main RAG chain for the synthesized answer.
    answer = rag_chain.invoke(question)

    # Format the source list for clear display in the UI.
    sources = ""
    for i, doc in enumerate(docs, 1):
        page    = doc.metadata.get('page', '?')
        # Show a short preview (150 chars) of the chunk.
        preview = doc.page_content[:150].replace('\n', ' ')
        sources += f"**[{i}] Page {page}**\n{preview}...\n\n"

    return answer, sources

# Setup Interface with 1 Input and 2 Outputs.
demo_sources = gr.Interface(
    fn=ask_newton_with_sources,
    inputs=gr.Textbox(
        label="Your Question",
        placeholder="Ask anything about Isaac Newton...",
        lines=2
    ),
    outputs=[
        # Define two output boxes to receive the 'return answer, sources_text'.
        gr.Textbox(label="💬 Answer", lines=6),
        gr.Textbox(label="📚 Retrieved Sources", lines=10),
    ],
    title="🍎 Newton Biography RAG + Sources",
    description="Ask questions about Isaac Newton — see retrieved sources",
    examples=[
        "When and where was Newton born?",
        "What did Newton discover during the plague years?",
        "What was Newton's relationship with Hooke?",
        "What was Newton's diet and daily routine?", # Check hallucination
    ]
)

demo_sources.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

## 5.3 Chat Interface with History

In [45]:

def chat_with_newton(message, history):

    # Skip processing if the message is empty.
    if not message.strip():
        return "", history

    # Invoke the RAG Chain to find the answer
    answer = rag_chain.invoke(message)

    # Append the current QA pair to the history list
    history.append([message, answer])

    # Return an empty string to clear the input box and update the chat history.
    return "", history


# gr.Blocks allows for flexible and custom UI layout design.
with gr.Blocks(title="🍎 Newton Biography Chat") as demo_chat:

    gr.Markdown("# 🍎 Newton Biography Chat\nAsk anything about Isaac Newton")

    # Chatbot Component: Displays the conversation thread.
    chatbot  = gr.Chatbot(height=400)

    # Input area for user questions.
    msg_box  = gr.Textbox(placeholder="Ask about Newton...", label="Your Question")

    # Arrange buttons in a single row.
    with gr.Row():
        submit_btn = gr.Button("Send", variant="primary")
        clear_btn  = gr.Button("Clear")

    # Add example questions for easy testing.
    gr.Examples(
        examples=[
            "When and where was Newton born?",
            "What did Newton discover during the plague years?",
            "What was Newton's relationship with Hooke?",
        ],
        inputs=msg_box
    )

    # On Click: Execute the chat function.
    submit_btn.click(chat_with_newton, [msg_box, chatbot], [msg_box, chatbot])

    # On Enter: Allow 'Enter' key to trigger submission.
    msg_box.submit(chat_with_newton,   [msg_box, chatbot], [msg_box, chatbot])

    # n Clear: Reset both history and input box using a lambda function.
    clear_btn.click(lambda: ([], ""),  outputs=[chatbot, msg_box])

# Launch the chat application.
demo_chat.launch(share=True)

/tmp/ipykernel_346/562668996.py:23: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot  = gr.Chatbot(height=400)
/tmp/ipykernel_346/562668996.py:23: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot  = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://954b6759265525e509.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
